# xbrl-frontier Phase 2: merge + INT8 W8A8 + AWQ-INT4 export of a trained student (owner-run)

Stages run in order in one session: **merge** (transformers + peft), **int8** (llm-compressor), **awq** (AutoAWQ in its own virtualenv), with AWQ calibrated on a fixed seeded sample of training instances (`serving.export_quantization.awq_calibration`).
**Settings:** GPU T4, Internet on. **Attach:** `xbrl-frontier-src`, `xbrl-frontier-corpus`, and the training run's output dataset (for `final_adapter/`).
**Afterwards:** save the output as a dataset, e.g. `xbrl-frontier-export-1p7b-s1`.

In [ ]:
MODEL = 'Qwen/Qwen3-1.7B'
ADAPTER = '/kaggle/input/xbrl-frontier-run-1p7b-s1/final_adapter'

In [ ]:
import glob, os, shutil, subprocess, sys
dst = '/kaggle/working/xbrl-frontier'
tarballs = glob.glob('/kaggle/input/**/xbrl-frontier-src.tar.gz', recursive=True)
trees = [os.path.dirname(os.path.dirname(os.path.dirname(p)))
         for p in glob.glob('/kaggle/input/**/docs/config/thresholds.yaml', recursive=True)]
if tarballs:
    assert len(tarballs) == 1, tarballs
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['tar', '-xzf', tarballs[0], '-C', dst], check=True)
else:  # Kaggle auto-extracted the upload
    assert len(trees) == 1, trees
    shutil.copytree(trees[0], dst, dirs_exist_ok=True)
os.chdir(dst)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import yaml
pins = yaml.safe_load(open('docs/config/thresholds.yaml'))['pins']['software']
missing = [k for k in ('transformers', 'peft', 'accelerate', 'llmcompressor', 'autoawq') if not pins.get(k)]
assert not missing, f'pins.software empty for {missing}: owner approval needed first'
print(pins)

In [ ]:
import glob, os
corpus = os.path.dirname(glob.glob('/kaggle/input/**/train.jsonl', recursive=True)[0])
out = f'/kaggle/working/export-{MODEL.split("/")[-1]}'
!pip install -q "transformers=={pins['transformers']}" "peft=={pins['peft']}" "accelerate=={pins['accelerate']}" "pyyaml==6.0.3" "pyarrow==25.0.1" "huggingface_hub"
import importlib.metadata as md, importlib
keys = ('transformers', 'peft', 'accelerate')
bad = {k: (md.version(k), pins[k]) for k in keys if md.version(k) != pins[k]}
assert not bad, f'installed != pins.software: {bad}'
print({k: md.version(k) for k in keys})
!python -m serving.export_quantized merge --model {MODEL} --adapter {ADAPTER} --out {out}

In [ ]:
!pip install -q "llmcompressor=={pins['llmcompressor']}"
import importlib.metadata as md, importlib
keys = ('llmcompressor',)
bad = {k: (md.version(k), pins[k]) for k in keys if md.version(k) != pins[k]}
assert not bad, f'installed != pins.software: {bad}'
print({k: md.version(k) for k in keys})
!python -m serving.export_quantized int8 --model {MODEL} --out {out}

In [ ]:
!pip install -q "virtualenv==20.29.3" && python -m virtualenv -q /kaggle/working/awqenv
!/kaggle/working/awqenv/bin/pip install -q "autoawq=={pins['autoawq']}" "pyyaml==6.0.3" "pyarrow==25.0.1" huggingface_hub
!/kaggle/working/awqenv/bin/python -c "import importlib.metadata as m; v=m.version('autoawq'); assert v=='{pins['autoawq']}', v; print('autoawq', v)"
!/kaggle/working/awqenv/bin/python -m serving.export_quantized awq --model {MODEL} --corpus {corpus} --out {out}

In [ ]:
!ls -la {out} && python -c "import json; [print(r['stage'], r.get('output'), list(r.get('output_hashes', {}))[:3]) for r in json.load(open('{out}/export.json'))]"